### Compare Flooring treatments with tuned XGBoost

This notebook keeps the experiment narrow. The baseline uses the loaded dataset’s modeling features with `Flooring` removed. The other three variants start from the same loaded data and change only `Flooring` (plus one numeric missingness flag): raw values with a flag, hierarchical mode imputation, and predictive imputation. All four use the same time split and XGBoost settings.

The hierarchical and predictive methods impute the original composite Flooring value as one categorical string. They do not expand the column into a large set of new material features.


In [1]:
# Imports and configuration
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("Install XGBoost in this notebook environment with: %pip install xgboost") from exc

import sklearn.preprocessing as sklearn_preprocessing
import idx_scripts.dataset_a as da
from idx_scripts.modeling import build_model, fit_and_score

warnings.filterwarnings("once", category=UserWarning, module="sklearn")
pd.set_option("display.float_format", "{:,.4f}".format)

TEST_MONTH = "2026-04"
TRAINING_WINDOW = 18
DATA_PATH = Path("../Rohan/CRMLSSoldDatasetA.csv")
RESULTS_DIR = Path("../Rohan/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# Compatibility with sklearn versions that require TargetEncoder(cv=integer).
# dataset_a.py constructs TargetEncoder with a KFold object.
if not getattr(da, "_target_encoder_compat_installed", False):
    class NotebookTargetEncoder(TargetEncoder):
        def __init__(self, categories="auto", target_type="auto", smooth="auto",
                     cv=5, shuffle=True, random_state=None):
            if not isinstance(cv, (int, np.integer)):
                splitter = cv
                cv = int(getattr(splitter, "n_splits", 5))
                shuffle = getattr(splitter, "shuffle", shuffle)
                random_state = getattr(splitter, "random_state", random_state)
            super().__init__(categories=categories, target_type=target_type,
                             smooth=smooth, cv=cv, shuffle=shuffle,
                             random_state=random_state)

    sklearn_preprocessing.TargetEncoder = NotebookTargetEncoder
    da.TargetEncoder = NotebookTargetEncoder
    da._target_encoder_compat_installed = True


In [3]:
# Flooring and dtype helpers. Missing/See Remarks-only values are treated as unavailable.
def flooring_is_missing(value):
    if pd.isna(value):
        return True
    parts = [part.strip() for part in str(value).split(",") if part.strip()]
    usable = [part for part in parts if part.casefold() != "see remarks"]
    return len(usable) == 0


def normalize_flooring(value):
    if flooring_is_missing(value):
        return np.nan
    parts = {
        part.strip().casefold().title()
        for part in str(value).split(",")
        if part.strip() and part.strip().casefold() != "see remarks"
    }
    return ", ".join(sorted(parts)) if parts else np.nan


def sklearn_safe_frame(frame):
    """Replace pandas nullable scalars with ordinary NumPy-compatible values."""
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col].dtype):
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
        elif not pd.api.types.is_datetime64_any_dtype(out[col].dtype):
            values = out[col].astype(object)
            out[col] = values.where(pd.notna(values), np.nan)
    return out


In [4]:
def hierarchical_impute_flooring(train, other, min_counts=(10, 20, 30)):
    """Fill missing composite Flooring values from training-only geographic modes."""
    train_out, other_out = train.copy(), other.copy()
    for frame in (train_out, other_out):
        frame["Flooring__missing"] = frame["Flooring"].map(
            lambda value: float(flooring_is_missing(value))
        )
        frame["Flooring"] = frame["Flooring"].map(normalize_flooring)
        year = pd.to_numeric(frame.get("YearBuilt", pd.Series(np.nan, index=frame.index)), errors="coerce")
        frame["_FlooringDecade"] = year.map(
            lambda value: int(value // 10 * 10) if pd.notna(value) else np.nan
        )

    known = train_out.loc[train_out["Flooring"].notna()]
    global_mode = known["Flooring"].mode().iloc[0] if not known.empty else np.nan
    levels = [
        (["PostalCode", "_FlooringDecade"], min_counts[0]),
        (["City", "_FlooringDecade"], min_counts[1]),
        (["CountyOrParish", "_FlooringDecade"], min_counts[2]),
    ]
    lookup_levels = []
    for keys, min_count in levels:
        if not all(key in known.columns for key in keys):
            continue
        counts = {}
        for row in known[keys + ["Flooring"]].itertuples(index=False, name=None):
            key_values, label = row[:-1], row[-1]
            if any(pd.isna(value) for value in key_values):
                continue
            counts.setdefault(tuple(key_values), {}).setdefault(label, 0)
            counts[tuple(key_values)][label] += 1
        lookup = {}
        for key, label_counts in counts.items():
            n = sum(label_counts.values())
            if n >= min_count:
                lookup[key] = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
        lookup_levels.append((keys, lookup))

    for frame in (train_out, other_out):
        for idx in frame.index[frame["Flooring"].isna()]:
            imputed = None
            for keys, lookup in lookup_levels:
                key = tuple(frame.at[idx, col] for col in keys)
                if any(pd.isna(value) for value in key):
                    continue
                imputed = lookup.get(key)
                if imputed is not None:
                    break
            frame.at[idx, "Flooring"] = imputed if imputed is not None else global_mode
        frame.drop(columns="_FlooringDecade", inplace=True)
    return train_out, other_out


In [5]:
def predictive_impute_flooring(train, other, predictor_features):
    """Predict the recorded composite Flooring label from training rows with known labels."""
    train_out, other_out = train.copy(), other.copy()
    for frame in (train_out, other_out):
        frame["Flooring__missing"] = frame["Flooring"].map(
            lambda value: float(flooring_is_missing(value))
        )
        frame["Flooring"] = frame["Flooring"].map(normalize_flooring)

    known = train_out["Flooring"].notna()
    missing_train = train_out["Flooring"].isna()
    missing_other = other_out["Flooring"].isna()
    if not (missing_train.any() or missing_other.any()) or not known.any():
        return train_out, other_out

    x_cols = [col for col in predictor_features if col in train.columns and col in other.columns]
    if not x_cols:
        raise ValueError("No baseline feature columns are available for predictive Flooring imputation.")
    x_known = sklearn_safe_frame(train.loc[known, x_cols])
    x_train_missing = sklearn_safe_frame(train.loc[missing_train, x_cols]) if missing_train.any() else None
    x_other_missing = sklearn_safe_frame(other.loc[missing_other, x_cols]) if missing_other.any() else None
    numeric_cols = x_known.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [col for col in x_cols if col not in numeric_cols]
    transformers = []
    if numeric_cols:
        transformers.append(("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_cols))
    if categorical_cols:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("categorical", categorical_pipe, categorical_cols))
    encoder = ColumnTransformer(transformers, remainder="drop")
    x_known_encoded = encoder.fit_transform(x_known)
    y = train_out.loc[known, "Flooring"].astype(str)

    if y.nunique() == 1:
        train_predictions = np.repeat(y.iloc[0], int(missing_train.sum()))
        other_predictions = np.repeat(y.iloc[0], int(missing_other.sum()))
    else:
        model = RandomForestClassifier(
            n_estimators=150, min_samples_leaf=5,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42,
        )
        model.fit(x_known_encoded, y)
        train_predictions = model.predict(encoder.transform(x_train_missing)) if x_train_missing is not None else []
        other_predictions = model.predict(encoder.transform(x_other_missing)) if x_other_missing is not None else []

    if missing_train.any():
        train_out.loc[missing_train, "Flooring"] = train_predictions
    if missing_other.any():
        other_out.loc[missing_other, "Flooring"] = other_predictions
    return train_out, other_out


In [6]:
# Load the prepared dataset and use its declared modeling features as the baseline.
df = da.load_clean(str(DATA_PATH))
if "Flooring" not in df.columns:
    raise ValueError(
        "The loaded CSV has no Flooring column. The current Library copy of dataset_a.py "
        "drops Flooring from FEATURES; make sure the CSV was created with Flooring retained."
    )

feature_candidates = list(getattr(da, "FEATURES", []))
BASE_FEATURES = [col for col in feature_candidates if col != "Flooring" and col in df.columns]
missing_base = [col for col in feature_candidates if col != "Flooring" and col not in df.columns]
if missing_base:
    raise ValueError(f"Declared baseline features are missing from the loaded CSV: {missing_base}")
if not BASE_FEATURES:
    raise ValueError("No baseline features were found in the loaded dataset.")

print(f"Rows: {len(df):,}; baseline feature count: {len(BASE_FEATURES)}")
print(f"Flooring values available: {df['Flooring'].notna().sum():,}")


Rows: 309,019; baseline feature count: 17
Flooring values available: 198,564


In [7]:
# The current dataset_a.py API does not accept categorical_cols=... .
# This small subclass lets the baseline omit Flooring while flooring variants encode it.
class NotebookDatasetAPreprocessor(da.DatasetAPreprocessor):
    def __init__(self, include_flooring=False, extra_numeric=(), random_state=0):
        super().__init__(extra_numeric=extra_numeric, extra_categorical=(), random_state=random_state)
        self.include_flooring = include_flooring

    def _categorical_cols(self):
        columns = [col for col in super()._categorical_cols() if col != "Flooring"]
        if self.include_flooring:
            columns.append("Flooring")
        return list(dict.fromkeys(columns))


def xgboost_regressor(**overrides):
    params = {
        "objective": "reg:squarederror", "tree_method": "hist",
        "n_estimators": 600, "max_depth": 8, "learning_rate": 0.1,
        "subsample": 0.8, "colsample_bytree": 0.8, "n_jobs": -1, "random_state": 42,
    }
    params.update(overrides)
    return XGBRegressor(**params)

# Replace with the selected parameters from your tuned XGBoost run.
BEST_XGB_PARAMS = {"max_depth": 8, "learning_rate": 0.1}


def make_model(include_flooring=False, include_missing_flag=False):
    extra_numeric = ["Flooring__missing"] if include_missing_flag else []
    prep = NotebookDatasetAPreprocessor(
        include_flooring=include_flooring,
        extra_numeric=extra_numeric,
        random_state=0,
    )
    return build_model(xgboost_regressor(**BEST_XGB_PARAMS), preprocessor=prep)


### Compare the four approaches

Every approach uses the same training rows, test rows, and tuned XGBoost settings. Only the `Flooring` column and its missingness flag differ. Imputation is learned from the training rows and then applied to both training and test rows.

In [8]:
train, test = da.split(df, TEST_MONTH, TRAINING_WINDOW)

def score_variant(name, train_variant, test_variant, features, include_flooring, include_missing_flag):
    # Convert pandas nullable extension dtypes (including string and Int64) before sklearn/XGBoost.
    model_features = list(features)
    train_input = sklearn_safe_frame(train_variant.loc[:, model_features + [da.TARGET]])
    test_input = sklearn_safe_frame(test_variant.loc[:, model_features + [da.TARGET]])
    model = make_model(include_flooring=include_flooring,
                       include_missing_flag=include_missing_flag)
    fitted, _, scores = fit_and_score(
        model, train_input, test_input, features=model_features, target=da.TARGET
    )
    predictions = fitted.predict(test_input.loc[:, model_features])
    return {
        "variant": name,
        "test_mae": scores["mae"],
        "test_rmse": mean_squared_error(test_input[da.TARGET], predictions) ** 0.5,
        "test_r2": scores["r2"],
        "test_mape": scores["mape"],
    }

rows = []
# 1) Baseline: use loaded dataset's feature list, explicitly excluding Flooring.
rows.append(score_variant(
    "Baseline",
    train, test, BASE_FEATURES,
    include_flooring=False, include_missing_flag=False,
))

# 2) Original Flooring values unchanged, with a separate missingness flag.
train_raw, test_raw = train.copy(), test.copy()
for frame in (train_raw, test_raw):
    frame["Flooring__missing"] = frame["Flooring"].map(
        lambda value: float(flooring_is_missing(value))
    )
rows.append(score_variant(
    "Raw Flooring + missingness",
    train_raw, test_raw, BASE_FEATURES + ["Flooring", "Flooring__missing"],
    include_flooring=True, include_missing_flag=True,
))

# 3) Hierarchical imputation: modify only Flooring and its missingness flag.
train_hier, test_hier = hierarchical_impute_flooring(train, test)
rows.append(score_variant(
    "Hierarchical imputation",
    train_hier, test_hier, BASE_FEATURES + ["Flooring", "Flooring__missing"],
    include_flooring=True, include_missing_flag=True,
))

# 4) Predictive imputation: predict the composite Flooring label from baseline features.
train_pred, test_pred = predictive_impute_flooring(train, test, BASE_FEATURES)
rows.append(score_variant(
    "Predictive imputation",
    train_pred, test_pred, BASE_FEATURES + ["Flooring", "Flooring__missing"],
    include_flooring=True, include_missing_flag=True,
))

comparison = pd.DataFrame(rows).sort_values("test_mae").reset_index(drop=True)
comparison


,variant,test_mae,test_rmse,test_r2,test_mape
0,Predictive imputation,"154,378.8663","304,566.4910",0.9041,0.1168
1,Raw Flooring + missingness,"155,308.3312","309,589.1887",0.9010,0.1171
2,Hierarchical imputation,"155,395.4595","308,882.0750",0.9014,0.1174
3,Baseline,"156,244.1041","309,433.3228",0.9011,0.1178


In [9]:
comparison.to_csv(RESULTS_DIR / "xgboost_flooring_strategy_comparison.csv", index=False)


### Reading the comparison

Compare the three Flooring variants with the baseline on MAE and RMSE. The missingness indicator records whether Flooring was absent before imputation, including `See Remarks`-only values. If this split suggests a winner, confirm it on another time window before adding the method to the shared scripts.